# 03 — L2 NiO e_g² SIAM: FakeMarrakesh noise study

**Phase 3 deliverable.** Reads per-config sweep results from
`03_L2_sweep_results/`, runs Layer 6 (stack monotonicity M3 → M3+ZNE)
and Layer 7 (mitigation effectiveness — hard gate), and renders
the comparison bar chart + ZNE extrapolation curves.

**Configs swept:** no_mit, m3_only, zne_lin_135, m3_zne_lin_135,
m3_zne_exp_135, m3_zne_poly3_135, m3_zne_lin_123, m3_zne_lin_12345.


In [1]:
import sys
from pathlib import Path

# Jupyter sets cwd to the notebook directory; the project root is one level up.
_PROJECT_ROOT = Path.cwd().parent  # siam_vqe/, from notebooks/ cwd
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

import json

import matplotlib
matplotlib.use('Agg')  # non-interactive backend for nbconvert
import matplotlib.pyplot as plt
import numpy as np

sweep_dir = Path.cwd() / '03_L2_sweep_results'

manifest = json.loads((sweep_dir / '_manifest.json').read_text())
ed_energy: float = manifest['ed_energy']
noiseless_energy: float = manifest['noiseless_vqe_energy']
x_star: list[float] = manifest['x_star']

print(f'ED energy:        {ed_energy:.8f} eV')
print(f'noiseless VQE:    {noiseless_energy:.8f} eV')
print(f'|ΔE| noiseless:   {abs(noiseless_energy - ed_energy):.2e} eV')
print(f'sweep started:    {manifest["wall_started"]}')
print(f'sweep ended:      {manifest["wall_ended"]}')
print(f'configs in manifest: {list(manifest["configs"].keys())}')


ED energy:        -95.39999857 eV
noiseless VQE:    -95.39996588 eV
|ΔE| noiseless:   3.27e-05 eV
sweep started:    2026-05-25T11:03:16
sweep ended:      2026-05-25T11:36:46
configs in manifest: ['no_mit', 'm3_only', 'zne_lin_135', 'm3_zne_lin_135', 'm3_zne_exp_135', 'm3_zne_poly3_135', 'm3_zne_lin_123', 'm3_zne_lin_12345']


## Load per-config sweep results


In [2]:
configs: dict[str, dict] = {}

for name in manifest['configs']:
    p = sweep_dir / f'{name}.json'
    if not p.exists():
        print(f'  {name}: FILE MISSING — skipping')
        continue
    res = json.loads(p.read_text())
    configs[name] = res
    if res.get('status') == 'ok':
        print(f'  {name}: E = {res["energy"]:.6f} ± {res["std"]:.6f} eV')
    else:
        print(f'  {name}: ERROR — {res.get("error", "unknown")}')

n_ok = sum(1 for r in configs.values() if r.get('status') == 'ok')
n_err = len(configs) - n_ok
print(f'\nLoaded {len(configs)} configs: {n_ok} ok, {n_err} errors')


  no_mit: E = -80.042906 ± 1.769835 eV
  m3_only: E = -81.807586 ± 1.234668 eV
  zne_lin_135: E = -85.102198 ± 6.119992 eV
  m3_zne_lin_135: E = -86.089960 ± 6.128530 eV
  m3_zne_exp_135: E = -93.257543 ± 6.112107 eV
  m3_zne_poly3_135: E = -93.081631 ± 6.375629 eV
  m3_zne_lin_123: E = -88.880041 ± 3.587930 eV
  m3_zne_lin_12345: E = -86.618962 ± 4.057901 eV

Loaded 8 configs: 8 ok, 0 errors


## Layer 6: stack monotonicity (M3 → M3+ZNE)

Pass-or-flag (not a hard gate). Checks that adding ZNE on top of M3
does not regress the gap to ED relative to M3 alone.


In [3]:
from siam_vqe.analysis import check_stack_monotonicity

m3_ok = configs.get('m3_only', {}).get('status') == 'ok'
m3_zne_ok = configs.get('m3_zne_lin_135', {}).get('status') == 'ok'

if m3_ok and m3_zne_ok:
    l6 = check_stack_monotonicity(
        e_ed=ed_energy,
        e_m3=configs['m3_only']['energy'],
        e_m3_zne=configs['m3_zne_lin_135']['energy'],
    )
    print(f'Layer 6 (stack monotonicity): passed={l6.passed}')
    print(f'  gap_m3       = {l6.gap_m3:.6f} eV')
    print(f'  gap_m3+zne   = {l6.gap_m3_zne:.6f} eV')
    print(f'  notes: {l6.notes}')
else:
    print('Layer 6: SKIPPED — m3_only or m3_zne_lin_135 not ok')
    l6 = None


Layer 6 (stack monotonicity): passed=True
  gap_m3       = 13.592413 eV
  gap_m3+zne   = 9.310038 eV
  notes: M3+ZNE improves over M3 alone


## Layer 7: mitigation effectiveness (Phase 3 headline)

**Hard gate.** At least one mitigation config must have
|E_mit - E_ED| / |E_no_mit - E_ED| < 1.0.


In [4]:
from siam_vqe.analysis import check_mitigation_effectiveness

l7 = check_mitigation_effectiveness(configs, ed_energy=ed_energy)
print(f'Layer 7 (mitigation effectiveness): passed={l7.passed}')
print(f'  best_config:  {l7.best_config}')
print(f'  best_ratio:   {l7.best_ratio:.6f}')
print()
print('Per-config ratios (sorted, ascending):')
for name, ratio in sorted(l7.per_config_ratio.items(), key=lambda kv: kv[1]):
    print(f'  {name:30s}: {ratio:.6f}')

assert l7.passed, f'Layer 7 FAILED — best ratio {l7.best_ratio:.4f} >= 1.0'


Layer 7 (mitigation effectiveness): passed=True
  best_config:  m3_zne_exp_135
  best_ratio:   0.139509

Per-config ratios (sorted, ascending):
  m3_zne_exp_135                : 0.139509
  m3_zne_poly3_135              : 0.150964
  m3_zne_lin_123                : 0.424557
  m3_zne_lin_12345              : 0.571790
  m3_zne_lin_135                : 0.606237
  zne_lin_135                   : 0.670557
  m3_only                       : 0.885090


## Figure 1: mitigation comparison bar chart


In [5]:
from siam_vqe.analysis import compare_mitigations

fig_mit_png = str(Path.cwd().parent / 'figures' / '03_L2_mitigation_comparison.png')
fig_mit_pdf = str(Path.cwd().parent / 'figures' / '03_L2_mitigation_comparison.pdf')

fig1 = compare_mitigations(
    configs,
    ed_energy=ed_energy,
    title='L2 NiO e_g\u00b2 SIAM — mitigation comparison (FakeMarrakesh)',
)
fig1.savefig(fig_mit_png, dpi=150)
fig1.savefig(fig_mit_pdf)
plt.close(fig1)
print(f'Saved {fig_mit_png}')
print(f'Saved {fig_mit_pdf}')


Saved figures/03_L2_mitigation_comparison.png
Saved figures/03_L2_mitigation_comparison.pdf


## Figure 2: ZNE extrapolation curves


In [6]:
from siam_vqe.analysis import plot_zne_extrapolation_curves

fig_zne_png = str(Path.cwd().parent / 'figures' / '03_L2_zne_curves.png')
fig_zne_pdf = str(Path.cwd().parent / 'figures' / '03_L2_zne_curves.pdf')

diagnostics: dict[str, dict] = {}
for name, res in configs.items():
    if res.get('status') != 'ok':
        continue
    md = res.get('metadata', {})
    if 'zne_raw_values' in md:
        diagnostics[name] = {
            'noise_factors': md['zne_noise_factors'],
            'raw_values': md['zne_raw_values'],
            'extrapolator': md['zne_extrapolator'],
            'extrapolated': res['energy'],
        }

print(f'ZNE diagnostics populated for {len(diagnostics)} configs:')
for name in diagnostics:
    print(f'  {name}')

fig2 = plot_zne_extrapolation_curves(
    diagnostics,
    title='L2 NiO e_g\u00b2 SIAM — ZNE extrapolation curves (FakeMarrakesh)',
)
fig2.savefig(fig_zne_png, dpi=150)
fig2.savefig(fig_zne_pdf)
plt.close(fig2)
print(f'Saved {fig_zne_png}')
print(f'Saved {fig_zne_pdf}')


ZNE diagnostics populated for 6 configs:
  zne_lin_135
  m3_zne_lin_135
  m3_zne_exp_135
  m3_zne_poly3_135
  m3_zne_lin_123
  m3_zne_lin_12345


siam_vqe/analysis.py:552: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(


Saved figures/03_L2_zne_curves.png
Saved figures/03_L2_zne_curves.pdf


## Write 03_L2_summary.json


In [7]:
summary = {
    'phase': 3,
    'level': 'L2',
    'ed_energy': ed_energy,
    'noiseless_vqe_energy': noiseless_energy,
    'best_mitigation_config': l7.best_config,
    'best_mitigation_ratio': l7.best_ratio,
    'layer_7_passed': bool(l7.passed),
    'layer_7_per_config_ratio': l7.per_config_ratio,
    'sweep_dir': str(sweep_dir),
    'sweep_manifest_started': manifest['wall_started'],
    'sweep_manifest_ended': manifest['wall_ended'],
}

summary_path = sweep_dir.parent / '03_L2_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print(f'Wrote {summary_path}')
print(json.dumps(summary, indent=2))


Wrote notebooks/03_L2_summary.json
{
  "phase": 3,
  "level": "L2",
  "ed_energy": -95.3999985731477,
  "noiseless_vqe_energy": -95.3999658761328,
  "best_mitigation_config": "m3_zne_exp_135",
  "best_mitigation_ratio": 0.13950920509462117,
  "layer_7_passed": true,
  "layer_7_per_config_ratio": {
    "m3_only": 0.8850902658021481,
    "zne_lin_135": 0.670556687426344,
    "m3_zne_lin_135": 0.6062370562829725,
    "m3_zne_exp_135": 0.13950920509462117,
    "m3_zne_poly3_135": 0.15096398436213904,
    "m3_zne_lin_123": 0.4245568023336721,
    "m3_zne_lin_12345": 0.5717903236368092
  },
  "sweep_dir": "notebooks/03_L2_sweep_results",
  "sweep_manifest_started": "2026-05-25T11:03:16",
  "sweep_manifest_ended": "2026-05-25T11:36:46"
}
